In [ ]:
import gmsh
import numpy as np

gmsh.initialize()
gmsh.model.add("wire_in_iron")

# =====================================================
# Geometry parameters
# =====================================================

L = 1.0

wire_r = 0.05

iron_w = 0.40
iron_h = 0.40

cx = 0.5
cy = 0.5

# =====================================================
# Outer air box
# =====================================================

outer = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    0.0,
    L,
    L
)

# =====================================================
# Iron block
# =====================================================

iron = gmsh.model.occ.addRectangle(
    cx - iron_w/2,
    cy - iron_h/2,
    0,
    iron_w,
    iron_h
)

# =====================================================
# Wire
# =====================================================

wire = gmsh.model.occ.addDisk(
    cx,
    cy,
    0,
    wire_r,
    wire_r
)

gmsh.model.occ.synchronize()

# =====================================================
# Fragment geometry
# =====================================================


gmsh.model.occ.fragment(
    [(2, outer)],
    [(2, iron), (2, wire)]
)

gmsh.model.occ.synchronize()

# =====================================================
# Identify regions
# =====================================================

# Fragment suffles the model tags for air,iron and wire. Lets use their unique area (center of mass funcitonality) to figure out the tags 

surfaces = gmsh.model.occ.getEntities(dim=2)

tol = 1e-6

# Areas
wire = np.pi * wire_r**2
iron = iron_w * iron_h - wire
air = L * L - iron_w * iron_h

air_region = []
iron_region = []
wire_region = []

for dim, tag in surfaces:

    area = gmsh.model.occ.getMass(dim, tag)

    if abs(area - wire) < tol:
        wire_region.append(tag)

    elif abs(area - iron) < tol:
        iron_region.append(tag)

    elif abs(area - air) < tol:
        air_region.append(tag)


# =====================================================
# Physical groups
# =====================================================

gmsh.model.addPhysicalGroup(2, air_region, name="air")
gmsh.model.addPhysicalGroup(2, iron_region, name="iron")
gmsh.model.addPhysicalGroup(2, wire_region, name="wire")

# Boundary
curves = gmsh.model.getEntities(1)

gmsh.model.addPhysicalGroup(
    1,
    [c[1] for c in curves],
    name="boundary"
)

# check if worked
print("Air:", air_region)
print("Iron:", iron_region)
print("Wire:", wire_region)

# =====================================================
# Mesh size
# =====================================================

gmsh.option.setNumber("Mesh.MeshSizeMin", 0.01)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.03)

gmsh.model.mesh.generate(2)

gmsh.write("wire_in_iron.msh")

gmsh.fltk.run()

gmsh.finalize()

TypeError: object of type 'float' has no len()